# Gemini Crypto Data Downloader

Downloads public Gemini candle data and stores it by symbol under `./data/<symbol>/`.

In [7]:
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests


In [8]:
# ---- Config ----
BASE_URL = "https://api.gemini.com"
SYMBOLS = ["btcusd", "ethusd", "solusd"]  # Gemini symbols are lowercase
TIMEFRAMES = ["1m", "5m", "1h", "1day"]
OUTPUT_ROOT = Path("./data")
SAVE_PARQUET = True
REQUEST_TIMEOUT = 20


In [ ]:
import time
import csv
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, List
from zoneinfo import ZoneInfo
import requests

BASE_URL = "https://api.gemini.com"
# BASE_URL = "https://api.sandbox.gemini.com"
EST_TZ = "America/New_York"

# NOTE: Gemini's public candles API returns at most 1440 candles per call
# (24 hours of 1-minute data). The `timestamp` parameter is ignored for 1m
# candles — it always returns the most recent 1440 bars.
# To maintain continuous data, run this script at least every 24 hours.
# Gaps larger than 1440 minutes cannot be backfilled via this API.
CANDLES_PER_CALL = 1440  # maximum returned by Gemini for 1m timeframe

TF_MS = {
    "1s": 1_000,
    "5s": 5_000,
    "30s": 30_000,
    "1m": 60_000,
    "5m": 5 * 60_000,
    "15m": 15 * 60_000,
    "30m": 30 * 60_000,
    "1h": 60 * 60_000,
    "6h": 6 * 60 * 60_000,
    "1d": 24 * 60 * 60_000,
}

def gemini_get_json(session: requests.Session, path: str, params=None, max_retries: int = 8):
    url = BASE_URL + path
    backoff = 0.5
    last_exc = None
    for _ in range(max_retries):
        try:
            r = session.get(url, params=params or {}, timeout=30)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(backoff)
                backoff = min(backoff * 2, 10.0)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_exc = e
            time.sleep(backoff)
            backoff = min(backoff * 2, 10.0)
    raise RuntimeError(f"Failed GET {url}") from last_exc

def list_symbols(session: requests.Session) -> List[str]:
    return gemini_get_json(session, "/v1/symbols")

def latest_closed_timestamp_ms(timeframe: str) -> int:
    tf = TF_MS[timeframe]
    now_ms = int(time.time() * 1000)
    return (now_ms // tf) * tf - tf  # last fully closed candle

def get_last_timestamp_ms(file_path: Path) -> int:
    if not file_path.exists() or file_path.stat().st_size == 0:
        return -1

    # fast tail-read
    with file_path.open("rb") as f:
        f.seek(0, 2)
        size = f.tell()
        chunk = min(size, 8192)
        f.seek(-chunk, 2)
        data = f.read(chunk)

    for line in reversed(data.splitlines()):
        line = line.strip()
        if not line or line.startswith(b"timestamp_ms"):
            continue
        parts = line.split(b",")
        try:
            return int(parts[0])
        except Exception:
            continue

    # fallback: full scan
    last_ts = -1
    with file_path.open("r", newline="") as ftxt:
        reader = csv.reader(ftxt)
        for row in reader:
            if not row or row[0] == "timestamp_ms":
                continue
            try:
                last_ts = int(row[0])
            except Exception:
                pass
    return last_ts

def append_rows(file_path: Path, rows: List[List[float]]):
    is_new = (not file_path.exists()) or file_path.stat().st_size == 0
    file_path.parent.mkdir(parents=True, exist_ok=True)
    with file_path.open("a", newline="") as f:
        w = csv.writer(f)
        if is_new:
            w.writerow(["timestamp_ms", "open", "high", "low", "close", "volume"])
        w.writerows(rows)

def normalize_filter_sort(candles, min_ts_exclusive: int, max_ts_inclusive: int) -> List[List[float]]:
    out = []
    for row in candles:
        if not isinstance(row, (list, tuple)) or len(row) < 6:
            continue
        ts = int(row[0])
        if ts > min_ts_exclusive and ts <= max_ts_inclusive:
            out.append([ts, float(row[1]), float(row[2]), float(row[3]), float(row[4]), float(row[5])])
    out.sort(key=lambda x: x[0])
    return out

def timestamp_ms_to_est_str(ts_ms: int, timezone_name: str = EST_TZ) -> str:
    dt_utc = datetime.fromtimestamp(ts_ms / 1000.0, tz=timezone.utc)
    return dt_utc.astimezone(ZoneInfo(timezone_name)).strftime("%Y-%m-%d %H:%M:%S%z")

def write_est_sidecar(source_file: Path, est_suffix: str = "_est.data", timezone_name: str = EST_TZ) -> Path:
    est_path = source_file.with_name(f"{source_file.stem}{est_suffix}")
    with source_file.open("r", newline="") as f_in, est_path.open("w", newline="") as f_out:
        reader = csv.DictReader(f_in)
        fieldnames = ["timestamp_ms", "timestamp_est", "open", "high", "low", "close", "volume"]
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()
        for row in reader:
            ts = int(row["timestamp_ms"])
            writer.writerow(
                {
                    "timestamp_ms": ts,
                    "timestamp_est": timestamp_ms_to_est_str(ts, timezone_name=timezone_name),
                    "open": row["open"],
                    "high": row["high"],
                    "low": row["low"],
                    "close": row["close"],
                    "volume": row["volume"],
                }
            )
    return est_path

def update_symbol_file(
    session: requests.Session,
    symbol: str,
    timeframe: str,
    out_dir: Path,
    sleep_between: float = 0.1,
    write_est: bool = True,
) -> int:
    file_path = out_dir / f"{symbol}.data"
    last_ts = get_last_timestamp_ms(file_path)
    closed_ts = latest_closed_timestamp_ms(timeframe)

    tf_ms = TF_MS[timeframe]
    gap_min = (closed_ts - max(last_ts, 0)) / 60_000 if last_ts > 0 else None
    if gap_min is not None and gap_min > CANDLES_PER_CALL:
        print(f"  WARNING: gap of {gap_min:.0f} min exceeds API window ({CANDLES_PER_CALL} min). "
              f"Data from {gap_min - CANDLES_PER_CALL:.0f} min ago to {CANDLES_PER_CALL} min ago is lost.")

    if last_ts < closed_ts:
        candles = gemini_get_json(session, f"/v2/candles/{symbol}/{timeframe}")
        new_rows = normalize_filter_sort(candles, min_ts_exclusive=last_ts, max_ts_inclusive=closed_ts)
        if new_rows:
            append_rows(file_path, new_rows)
    else:
        new_rows = []

    if write_est and file_path.exists():
        write_est_sidecar(file_path)

    time.sleep(sleep_between)
    return len(new_rows)

def update_symbols_inplace(
    symbols: List[str],
    timeframe: str,
    out_dir: str,
    sleep_between: float = 0.1,
    write_est: bool = True,
):
    if timeframe not in TF_MS:
        raise ValueError(f"Unsupported timeframe {timeframe}. Choose from {list(TF_MS.keys())}")

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    with requests.Session() as session:
        (out_path / "symbols.txt").write_text("\n".join(symbols))

        total_new = 0
        for i, sym in enumerate(symbols, 1):
            try:
                n_new = update_symbol_file(
                    session,
                    sym,
                    timeframe,
                    out_path,
                    sleep_between=sleep_between,
                    write_est=write_est,
                )
                total_new += n_new
                print(f"[{i}/{len(symbols)}] {sym}: appended={n_new} total_appended={total_new}")
            except Exception as e:
                print(f"[{i}/{len(symbols)}] FAILED {sym}: {type(e).__name__}: {e}")

    print(f"Done. Updated files in-place at: {out_path.resolve()}")
    if write_est:
        print(f"Also refreshed EST sidecars as <symbol>_est.data in: {out_path.resolve()}")

if __name__ == "__main__":
    update_symbols_inplace(
        symbols=["btcusd", "ethusd", "solusd"],
        timeframe="1m",
        out_dir=".data/gemini/ohlcv_1m_7d",
        sleep_between=0.1,
        write_est=True,
    )


In [ ]:
# Convert and preview timestamp in EST for a single symbol file.
from pathlib import Path
import pandas as pd

source_file = Path('.data/gemini/ohlcv_1m_7d/btcusd.data')
est_file = write_est_sidecar(source_file)

preview = pd.read_csv(est_file).head(10)
preview

